# 04_04 · SPARK TEC — Detección de anomalías eléctricas en máquinas herramienta

**Dataset**: SPARK — High-Resolution Energy Data from a Sustainable Industrial Production Area in Karlsruhe  
**DOI**: 10.35097/bjdg3m3rg5jv3skk  
**Máquinas**: 11 máquinas TEC (fresadoras CNC, tornos, EDM) — año 2024  
**Resolución original**: 5 segundos → agregado a 1 minuto  
**Modelo**: Isolation Forest por máquina (contamination=5%)  

Variables usadas (12 señales × 3 estadísticos = 36 features):  
`P_total, P1, P2, P3` — potencia activa (W)  
`I1, I2, I3` — corriente de fase (A)  
`Freq` — frecuencia de red (Hz)  
`THD_I1, THD_I2, THD_I3` — distorsión armónica total (%)  
`PF_total` — factor de potencia  

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import os
import gc
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Paths
BASE = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
PROCESSED = os.path.join(BASE, 'data', 'processed')
RAW_TEC   = os.path.join(BASE, 'data', 'raw', 'tec_extracted')

MACHINES = sorted([
    'TEC_48S', 'TEC_CFST161', 'TEC_CTX800TC', 'TEC_Chiron800',
    'TEC_DMF3008', 'TEC_DMU125MB', 'TEC_DNG50evo', 'TEC_E110',
    'TEC_E30D2', 'TEC_JWA24', 'TEC_MV2400R'
])

VARS = ['P_total','P1','P2','P3','I1','I2','I3','Freq','THD_I1','THD_I2','THD_I3','PF_total']
FEATURE_COLS = [f'{v}_{s}' for v in VARS for s in ['mean','std','max']]

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('tab10')
print(f'Machines: {len(MACHINES)}')
print(f'Feature columns: {len(FEATURE_COLS)}')

## 1. Carga del dataset agregado

Los archivos de entrada son los parquets pre-procesados (`spark_{MACHINE}_scored.parquet`) generados en el notebook de preprocesado. Cada archivo contiene:
- Las 36 features estadísticas (mean/std/max por variable eléctrica)
- `is_anomaly`: `1` = anomalía, `0` = operación normal, `-1` = máquina apagada / gap de datos
- `anomaly_score`: valor continuo del `decision_function` de Isolation Forest — cuanto más negativo, más anómalo

Cargamos los datos ya puntuados en lugar de re-entrenar porque el modelo fue entrenado máquina a máquina con `contamination=0.05`, etiquetando el 5% de puntos más alejados como anomalías. Identificar la estructura del dataset antes de visualizar evita malinterpretar los gaps como anomalías.

In [ ]:
# Load pre-scored dataset (Isolation Forest already trained per machine)
dfs = {}
for m in MACHINES:
    path = os.path.join(PROCESSED, f'spark_{m}_scored.parquet')
    dfs[m] = pd.read_parquet(path)
    print(f'{m:<18} {len(dfs[m]):>8,} filas | '
          f'anomalías: {(dfs[m]["is_anomaly"]==1).sum():>6,} '
          f'({(dfs[m]["is_anomaly"]==1).mean():.1%}) | '
          f'gaps: {(dfs[m]["is_anomaly"]==-1).sum():>6,} '
          f'({(dfs[m]["is_anomaly"]==-1).mean():.1%})')

## 2. Visión general: consumo anual de potencia activa

Antes de cualquier análisis estadístico es fundamental ver los datos en su contexto temporal completo.

La serie anual de `P_total_mean` por máquina permite detectar:
- **Patrones estacionales**: caídas en agosto (paradas por vacaciones), repuntes en septiembre
- **Máquinas "ruidosas"** (anomalías dispersas durante todo el año) vs. **eventos puntuales** (ráfagas concentradas en una semana)
- **Gaps de apagado**: franjas planas en cero que el modelo etiqueta como `-1` y que aquí quedan fuera de los puntos rojos

Los puntos rojos son los minutos clasificados como anomalía. Se muestra a resolución horaria para legibilidad; el modelo trabaja a 1 minuto.

In [ ]:
fig, axes = plt.subplots(6, 2, figsize=(16, 20), sharex=True)
axes = axes.flatten()

for i, m in enumerate(MACHINES):
    ax = axes[i]
    df = dfs[m]
    
    # Resample to hourly for visibility
    hourly = df['P_total_mean'].resample('1h').mean()
    
    # Normal vs anomaly
    normal = df[df['is_anomaly'] == 0]['P_total_mean'].resample('1h').mean()
    anom   = df[df['is_anomaly'] == 1]['P_total_mean'].resample('1h').mean()
    
    ax.plot(hourly.index, hourly.values, color='steelblue', alpha=0.5, linewidth=0.4, label='P_total (W)')
    ax.scatter(anom.index, anom.values, color='red', s=2, alpha=0.6, label='anomalía', zorder=3)
    
    ax.set_title(m.replace('TEC_', ''), fontsize=10, fontweight='bold')
    ax.set_ylabel('P_total (W)', fontsize=8)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
    ax.tick_params(labelsize=7)
    if i == 0:
        ax.legend(fontsize=7, markerscale=3)

# Hide unused subplot
axes[-1].set_visible(False)
fig.suptitle('Potencia activa total (P_total) — TEC machines 2024\nRojo = anomalía detectada por Isolation Forest', 
             fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(PROCESSED, 'spark_tec_potencia_anual.png'), dpi=120, bbox_inches='tight')
plt.show()

## 3. Distribución del anomaly score por máquina

El `decision_function` de Isolation Forest devuelve un score continuo: **valores negativos = anomalía**, valores positivos = normal. El umbral natural es 0.

Analizar la distribución completa (no solo el binario `is_anomaly`) revela:
- **Distribuciones bimodales**: buena separación entre estados — el modelo ha aprendido una firma eléctrica real
- **Distribuciones unimodales centradas en 0**: el modelo no encuentra estructura clara; puede indicar que la `contamination` está mal ajustada o que esa máquina opera de forma muy homogénea
- **Colas largas en negativo**: episodios muy anómalos que merecen inspección detallada en el zoom de la sección 6

La línea roja discontinua en `score=0` marca la frontera de decisión del modelo.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

colors = plt.cm.tab20(np.linspace(0, 1, len(MACHINES)))

for i, m in enumerate(MACHINES):
    df = dfs[m]
    scores = df[df['anomaly_score'].notna()]['anomaly_score']
    ax.hist(scores, bins=100, alpha=0.5, label=m.replace('TEC_',''), 
            color=colors[i], density=True)

ax.axvline(0, color='red', linestyle='--', linewidth=1.5, label='umbral (score=0)')
ax.set_xlabel('Anomaly score (Isolation Forest decision function)', fontsize=11)
ax.set_ylabel('Densidad', fontsize=11)
ax.set_title('Distribución del anomaly score — valores negativos = anomalía', fontsize=12)
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

## 4. Análisis de anomalías por mes

Una vez confirmada la señal global, exploramos su **distribución temporal** a nivel mensual.

Concentraciones de anomalías en meses concretos pueden indicar:
- **Eventos de mantenimiento o reconfiguraciones** de máquina (cambio de herramienta, ajuste de parámetros)
- **Cambios de carga productiva**: más piezas, materiales distintos, mayor velocidad de avance
- **Degradación progresiva**: si las anomalías aumentan mes a mes de forma sostenida, hay un proceso de deterioro en curso

El heatmap normaliza por total de minutos válidos (excluye gaps con `is_anomaly == -1`), por lo que el porcentaje es comparable entre máquinas con distintas horas de operación anual.

In [ ]:
monthly_data = []
for m in MACHINES:
    df = dfs[m]
    valid = df[df['is_anomaly'] != -1]
    monthly = valid.resample('ME').agg(
        total=('is_anomaly', 'count'),
        anomalias=('is_anomaly', 'sum')
    )
    monthly['pct_anomalia'] = monthly['anomalias'] / monthly['total'] * 100
    monthly['machine'] = m.replace('TEC_','')
    monthly_data.append(monthly)

monthly_all = pd.concat(monthly_data)
pivot = monthly_all.pivot_table(index=monthly_all.index.month, 
                                 columns='machine', 
                                 values='pct_anomalia')
pivot.index = ['Ene','Feb','Mar','Abr','May','Jun','Jul','Ago','Sep','Oct','Nov','Dic']

fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(pivot.T, annot=True, fmt='.1f', cmap='YlOrRd', 
            linewidths=0.5, ax=ax, cbar_kws={'label': '% anomalías'})
ax.set_title('Porcentaje de minutos anómalos por máquina y mes (2024)', fontsize=12)
ax.set_xlabel('Mes')
ax.set_ylabel('Máquina')
plt.tight_layout()
plt.savefig(os.path.join(PROCESSED, 'spark_tec_anomalias_mensuales.png'), dpi=120, bbox_inches='tight')
plt.show()

## 5. PCA — visualización de estados de máquina

Con 36 features es imposible visualizar directamente el espacio de datos. PCA proyecta en 2 componentes principales para hacer visible la geometría del problema.

Lo que buscamos:
- **Clusters separados** de anomalías en el espacio PCA → Isolation Forest captura una firma eléctrica consistente, no ruido aleatorio
- **Mezcla de colores** → las anomalías no tienen una firma diferenciada en estas dos dimensiones (puede estar en componentes superiores)
- **Gradiente suave de score** → el modelo asigna scores de forma coherente con la distancia al centro de masa de los datos normales

Un alto porcentaje de varianza en PC1+PC2 (>50%) indica que 2 dimensiones capturan bien la estructura. Si la varianza es baja, las conclusiones visuales deben tomarse con cautela.

In [ ]:
# PCA sobre una máquina de ejemplo: TEC_Chiron800
machine = 'TEC_Chiron800'
df = dfs[machine]

# Sample 20k points for speed
valid = df[df['is_anomaly'] != -1][FEATURE_COLS + ['is_anomaly', 'anomaly_score']].dropna()
sample = valid.sample(min(20000, len(valid)), random_state=42)

X = StandardScaler().fit_transform(sample[FEATURE_COLS])
pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(X)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: color by anomaly score
sc = axes[0].scatter(coords[:, 0], coords[:, 1], 
                     c=sample['anomaly_score'].values,
                     cmap='RdYlGn', s=1, alpha=0.5)
plt.colorbar(sc, ax=axes[0], label='anomaly score')
axes[0].set_title(f'{machine} — PCA coloreado por anomaly score', fontsize=10)
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} var)')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} var)')

# Right: normal vs anomaly
is_anom = sample['is_anomaly'].values == 1
axes[1].scatter(coords[~is_anom, 0], coords[~is_anom, 1], 
                c='steelblue', s=1, alpha=0.3, label='normal')
axes[1].scatter(coords[is_anom, 0], coords[is_anom, 1], 
                c='red', s=3, alpha=0.7, label='anomalía')
axes[1].set_title(f'{machine} — PCA: normal vs anomalía', fontsize=10)
axes[1].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} var)')
axes[1].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} var)')
axes[1].legend(markerscale=5, fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(PROCESSED, 'spark_tec_pca_chiron800.png'), dpi=120, bbox_inches='tight')
plt.show()

print(f'Varianza explicada PC1+PC2: {pca.explained_variance_ratio_[:2].sum():.1%}')

## 6. Zoom: episodios anómalos en detalle

Las vistas anuales ocultan la **microestructura** de los episodios. El zoom a nivel de semana permite distinguir:
- **Anomalías aisladas** (puntos rojos dispersos): posiblemente ruido de medición o transitorios breves sin impacto real
- **Ráfagas consecutivas** de varios minutos: indican un evento sostenido — sobrecarga, vibración, problema de calidad de red eléctrica
- **Firmas multivariadas**: si varias señales (`P_total`, `THD`, `PF`) se disparan simultáneamente, el evento es más grave y creíble que si solo sube una variable

Seleccionamos automáticamente la semana con mayor densidad de anomalías para maximizar la información visible en el análisis.

In [ ]:
def plot_anomaly_episode(machine, start, end, variables=['P_total_mean','I1_mean','THD_I1_mean','PF_total_mean']):
    df = dfs[machine].loc[start:end]
    
    fig, axes = plt.subplots(len(variables), 1, figsize=(14, len(variables)*2.2), sharex=True)
    if len(variables) == 1:
        axes = [axes]
    
    for ax, var in zip(axes, variables):
        normal = df[df['is_anomaly'] != 1][var]
        anom   = df[df['is_anomaly'] == 1][var]
        ax.plot(normal.index, normal.values, color='steelblue', linewidth=0.8, label='normal')
        ax.scatter(anom.index, anom.values, color='red', s=8, zorder=3, label='anomalía')
        ax.set_ylabel(var.replace('_mean',''), fontsize=9)
        ax.tick_params(labelsize=8)
    
    axes[0].legend(fontsize=8)
    axes[0].set_title(f'{machine} — {start} → {end}', fontsize=11)
    axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%d %b %H:%M'))
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()

# Encontrar una semana con alta densidad de anomalías
m = 'TEC_Chiron800'
weekly_anom = dfs[m][dfs[m]['is_anomaly']==1]['is_anomaly'].resample('W').count()
worst_week = weekly_anom.idxmax()
start_w = (worst_week - pd.Timedelta(days=6)).strftime('%Y-%m-%d')
end_w   = worst_week.strftime('%Y-%m-%d')
print(f'Semana con más anomalías en {m}: {start_w} → {end_w} ({weekly_anom.max()} anomalías)')

plot_anomaly_episode(m, start_w, end_w)

## 7. Feature importance — ¿qué variables impulsan las anomalías?

Isolation Forest no proporciona importancias nativas. Usamos una aproximación interpretable: **diferencia media normalizada** entre puntos anómalos y normales, expresada en unidades de desviación estándar (σ).

Una feature con `|diff| > 0.5σ` discrimina de forma relevante entre estados. Esto tiene valor operacional directo:
- **THD_I alto** → armónicos eléctricos → revisar variadores de frecuencia o cargas no lineales conectadas
- **PF_total bajo** → factor de potencia degradado → posible motor deteriorado o sobrecarga reactiva
- **I_mean elevado con P estable** → corriente reactiva alta → ineficiencia energética, riesgo de calentamiento

El análisis se hace sobre `TEC_Chiron800` como máquina representativa; la función es reutilizable para cualquier máquina cambiando la variable `machine`.

In [ ]:
# Mean absolute difference between anomalous and normal points, per feature
machine = 'TEC_Chiron800'
df = dfs[machine]

normal = df[df['is_anomaly'] == 0][FEATURE_COLS].dropna()
anomal = df[df['is_anomaly'] == 1][FEATURE_COLS].dropna()

# Normalized difference (z-score units)
scaler = StandardScaler().fit(normal)
diff = (scaler.transform(anomal).mean(axis=0) - scaler.transform(normal).mean(axis=0))
importance = pd.Series(np.abs(diff), index=FEATURE_COLS).sort_values(ascending=True)

# Plot top 20
top20 = importance.tail(20)
fig, ax = plt.subplots(figsize=(8, 7))
colors_bar = ['#e74c3c' if v > 0.5 else '#3498db' for v in top20.values]
top20.plot(kind='barh', ax=ax, color=colors_bar)
ax.axvline(0.5, color='gray', linestyle='--', linewidth=1, alpha=0.7)
ax.set_xlabel('|Diferencia media normalizada| (σ)', fontsize=10)
ax.set_title(f'{machine}\nFeatures que más distinguen anomalías de puntos normales', fontsize=11)
ax.tick_params(labelsize=8)
plt.tight_layout()
plt.savefig(os.path.join(PROCESSED, 'spark_tec_feature_importance.png'), dpi=120, bbox_inches='tight')
plt.show()

## 8. Comparativa de anomalías entre máquinas

El objetivo de este análisis es **priorizar intervenciones de mantenimiento**. No todas las máquinas merecen la misma atención.

Tres métricas clave:
1. **% minutos anómalos**: una máquina muy por encima del 5% de `contamination` no implica que el modelo falle — indica que esa máquina tiene un comportamiento estructuralmente distinto al periodo de referencia y merece inspección física
2. **Score medio**: cuanto más bajo (más negativo), más lejos están sus puntos del centro de masa normal — puede indicar alta variabilidad operativa o degradación acumulada
3. **Consumo medio (kW)**: contextualiza las anomalías — un pico en una máquina de 2 kW tiene un impacto energético muy diferente al de una de 20 kW

Las máquinas con alta tasa de anomalía **y** score medio muy negativo son las candidatas prioritarias a inspección física.

In [ ]:
summary = []
for m in MACHINES:
    df = dfs[m]
    valid = df[df['is_anomaly'] != -1]
    anom  = df[df['is_anomaly'] == 1]
    off   = df[df['is_anomaly'] == -1]
    summary.append({
        'machine': m.replace('TEC_', ''),
        'minutos_validos': len(valid),
        'minutos_anomalia': len(anom),
        'pct_anomalia': len(anom)/len(valid)*100,
        'minutos_gap': len(off),
        'pct_gap': len(off)/len(df)*100,
        'score_medio': valid['anomaly_score'].mean(),
        'P_total_media_kW': valid['P_total_mean'].mean()/1000,
    })

summary_df = pd.DataFrame(summary).set_index('machine')

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# 1. % anomalías
summary_df['pct_anomalia'].sort_values().plot(
    kind='barh', ax=axes[0], color='salmon')
axes[0].set_xlabel('% minutos anómalos')
axes[0].set_title('% anomalías por máquina')
axes[0].axvline(5, color='red', linestyle='--', linewidth=1, label='contamination=5%')
axes[0].legend(fontsize=8)

# 2. Score medio (más alto = más normal)
summary_df['score_medio'].sort_values().plot(
    kind='barh', ax=axes[1], color='steelblue')
axes[1].set_xlabel('Anomaly score medio')
axes[1].set_title('Score medio (mayor = más normal)')

# 3. Consumo medio kW
summary_df['P_total_media_kW'].sort_values().plot(
    kind='barh', ax=axes[2], color='mediumseagreen')
axes[2].set_xlabel('kW')
axes[2].set_title('Consumo medio P_total (kW)')

plt.suptitle('Resumen comparativo — TEC machines 2024', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(PROCESSED, 'spark_tec_comparativa.png'), dpi=120, bbox_inches='tight')
plt.show()

print(summary_df[['minutos_validos','minutos_anomalia','pct_anomalia','pct_gap','P_total_media_kW']]
      .round(2).to_string())

## 9. Conclusiones — SPARK TEC

- **Señal real, no ruido**: las distribuciones bimodales del anomaly score y la separación en PCA confirman que Isolation Forest captura estados eléctricos diferenciados, no variabilidad aleatoria
- **Heterogeneidad entre máquinas**: la tasa de anomalía varía de forma notable entre máquinas — cada una tiene un perfil operativo propio que justifica el entrenamiento individual en lugar de un modelo global
- **Estacionalidad presente**: las concentraciones mensuales de anomalías no son uniformes y correlacionan con paradas productivas, cambios de utillaje o reajustes de proceso
- **Variables discriminantes**: `THD_I` y `PF_total` son las features que más separan anomalías de operación normal — apuntan a problemas de calidad de red y eficiencia energética, no a sobrecarga mecánica
- **Episodios vs. ruido**: el zoom semanal distingue ráfagas reales (varios minutos consecutivos) de transitorios aislados — criterio práctico para filtrar falsas alarmas en producción

### Implicaciones para uso en producción
- Implementar un **filtro de persistencia**: solo alertar si la anomalía dura ≥ 3 minutos consecutivos
- Revisar `contamination` en máquinas con >7% de anomalías — puede ser necesario reajustar el parámetro o ampliar el periodo de referencia con datos más limpios
- Cruzar episodios SPARK con el **CMMS** (órdenes de trabajo y paradas registradas) para validar etiquetas y evolucionar el modelo hacia aprendizaje supervisado

## 10. Exportar resultados

In [ ]:
# Save summary table
summary_df.to_csv(os.path.join(PROCESSED, 'spark_tec_anomaly_summary.csv'))
print('Guardado: spark_tec_anomaly_summary.csv')

# The full scored dataset is already at:
# data/processed/spark_tec_scored.parquet   — todas las máquinas
# data/processed/spark_{MACHINE}_scored.parquet — por máquina
print('Dataset completo: data/processed/spark_tec_scored.parquet')
print(f'Columnas clave: {["machine","is_anomaly","anomaly_score"] + FEATURE_COLS[:3] + ["..."]}')

# Usage example
print('\n--- Ejemplo de carga ---')
print("df = pd.read_parquet('data/processed/spark_tec_scored.parquet')")
print("anomalias = df[df['is_anomaly'] == 1]")